# 05. Deployment and Consumption - MLflow Credit Risk

Este notebook documenta como transformar o resultado analitico em uma entrega consumivel.
O treino e as principais analises continuam reproduziveis no Colab; a interface do MLflow e os canais online devem ser executados em ambiente local ou em uma plataforma de deploy.


## Canais de entrega

- MLflow Tracking para comparar modelos, parametros, metricas e artefatos.
- Model Registry para promover o melhor modelo para `Production` quando houver tracking remoto configurado.
- API FastAPI para consumo online em `/predict`.
- Template SageMaker apenas como opcao enterprise.
- Alternativas de menor custo: DagsHub para tracking remoto e Render, Railway, Fly.io, Google Cloud Run ou Hugging Face Spaces para a API/demo.


## Como ver a interface do MLflow

Rode a UI em um terminal local a partir da raiz do repositorio e abra `http://127.0.0.1:5000`.
Depois execute o treino em outro terminal. A run aparecera no experimento `08_mlflow_credit_risk`.


In [ ]:
# Terminal 1: start the MLflow UI locally
# python -m pip install -r requirements.txt -r requirements-mlflow.txt
# mlflow ui --backend-store-uri ./mlruns --host 127.0.0.1 --port 5000

# Terminal 2: generate a tracked run
# PYTHONPATH=src python -m mlflow_credit_risk.cli train


## API local

A API usa o modelo salvo em `models/model.joblib`. Antes de consumir `/predict`, gere o artefato com o comando de treino.


In [ ]:
# Generate model artifacts
!PYTHONPATH=src python -m mlflow_credit_risk.cli train

# Local API
# !PYTHONPATH=src uvicorn mlflow_credit_risk.api:app --reload

# Sample API request
# !PYTHONPATH=src python scripts/sample_api_request.py


## Alternativas mais baratas que AWS

Para portfolio, o caminho de melhor custo costuma ser:

1. MLflow local ou DagsHub para mostrar tracking de experimentos.
2. API FastAPI publicada em Render, Railway, Fly.io ou Google Cloud Run.
3. Hugging Face Spaces se a entrega virar uma demo publica em Docker ou app visual.

SageMaker continua documentado no projeto, mas deve ser usado apenas quando o objetivo for demonstrar uma stack AWS gerenciada.


In [ ]:
# SageMaker dry run only; it does not create resources by default.
!python scripts/sagemaker_deployment_template.py --model-uri models:/mlflow-credit-risk/Production


## Leitura de negocio

A camada MLflow torna o projeto auditavel: negocio consegue saber qual modelo esta ativo, quais metricas justificaram a promocao e qual contrato de consumo sera usado.
A separacao entre tracking e inferencia reduz custo porque a API pode rodar em uma plataforma pequena, enquanto o historico de experimentos fica local ou em tracking remoto barato.
